In [1]:
import torch

print("CUDA:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

x = torch.randn(100, 100, device="cuda")
print("CUDA tensor test passed:", x.device)

CUDA: True
GPU count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4
CUDA tensor test passed: cuda:0


In [2]:
from pathlib import Path

DATASET_DIR = Path(
    "/kaggle/input/datasets/krishnans2005/drivealert-processed-3"
)

In [3]:
!find /kaggle/input/datasets/krishnans2005/drivealert-processed-3 \
  -maxdepth 3 -type f | head -30


/kaggle/input/datasets/krishnans2005/drivealert-processed-3/splits/person_splits.csv
/kaggle/input/datasets/krishnans2005/drivealert-processed-3/splits/weak_label_metadata.json
/kaggle/input/datasets/krishnans2005/drivealert-processed-3/splits/eye_val_provisional.csv
/kaggle/input/datasets/krishnans2005/drivealert-processed-3/splits/eye_train_provisional.csv
/kaggle/input/datasets/krishnans2005/drivealert-processed-3/splits/split_summary.csv
/kaggle/input/datasets/krishnans2005/drivealert-processed-3/splits/eye_test_provisional.csv
/kaggle/input/datasets/krishnans2005/drivealert-processed-3/splits/mouth_test_provisional.csv
/kaggle/input/datasets/krishnans2005/drivealert-processed-3/splits/mouth_val_provisional.csv
/kaggle/input/datasets/krishnans2005/drivealert-processed-3/splits/mouth_train_provisional.csv
/kaggle/input/datasets/krishnans2005/drivealert-processed-3/PACKAGE_CONTENTS.json
/kaggle/input/datasets/krishnans2005/drivealert-processed-3/README.md
/kaggle/input/datasets/krish

In [5]:
from pathlib import Path
import os
import json
import torch
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

DATASET_DIR = Path(
    "/kaggle/input/datasets/krishnans2005/drivealert-processed-3"
)
OUTPUT_DIR = Path("/kaggle/working")
SPLIT_DIR = DATASET_DIR / "splits"

EYE_LABELS = {
    "open": 0,
    "closed": 1,
}

def load_eye_split(split):
    csv_path = SPLIT_DIR / f"eye_{split}_provisional.csv"
    df = pd.read_csv(csv_path)

    required = {
        "filepath", "label", "person_id",
        "video_id", "dataset", "is_ground_truth"
    }
    missing_columns = required - set(df.columns)
    assert not missing_columns, f"Missing columns: {missing_columns}"

    assert set(df["label"]) <= set(EYE_LABELS)
    assert not df["filepath"].duplicated().any()

    df["image_path"] = df["filepath"].map(
        lambda path: DATASET_DIR / path
    )

    missing_files = [
        path for path in df["image_path"]
        if not path.is_file()
    ]
    assert not missing_files, f"{len(missing_files)} images are missing"

    return df

train_df = load_eye_split("train")
val_df = load_eye_split("val")
test_df = load_eye_split("test")

train_people = set(train_df["person_id"])
val_people = set(val_df["person_id"])
test_people = set(test_df["person_id"])

assert train_people.isdisjoint(val_people)
assert train_people.isdisjoint(test_people)
assert val_people.isdisjoint(test_people)

print("Eye split verification passed")
print()
print("Train:", len(train_df), "images,", len(train_people), "drivers")
print(train_df["label"].value_counts().to_string())
print()
print("Validation:", len(val_df), "images,", len(val_people), "drivers")
print(val_df["label"].value_counts().to_string())
print()
print("Test:", len(test_df), "images,", len(test_people), "drivers")
print(test_df["label"].value_counts().to_string())

train_transform = transforms.Compose([
    transforms.Resize((96, 192), antialias=True),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomApply([
        transforms.ColorJitter(
            brightness=0.25,
            contrast=0.25,
            saturation=0.10
        )
    ], p=0.7),
    transforms.RandomAffine(
        degrees=5,
        translate=(0.03, 0.03),
        scale=(0.95, 1.05),
        fill=(128, 128, 128),
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225),
    ),
])

eval_transform = transforms.Compose([
    transforms.Resize((96, 192), antialias=True),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225),
    ),
])

class EyeDataset(Dataset):
    def __init__(self, dataframe, transform):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]

        with Image.open(row["image_path"]) as image:
            image = image.convert("RGB")
            image = self.transform(image)

        label = torch.tensor(
            EYE_LABELS[row["label"]],
            dtype=torch.float32
        )

        return image, label

train_ds = EyeDataset(train_df, train_transform)
val_ds = EyeDataset(val_df, eval_transform)
test_ds = EyeDataset(test_df, eval_transform)

workers = min(4, os.cpu_count() or 2)

train_loader = DataLoader(
    train_ds,
    batch_size=128,
    shuffle=True,
    num_workers=workers,
    pin_memory=True,
    persistent_workers=workers > 0,
)

val_loader = DataLoader(
    val_ds,
    batch_size=256,
    shuffle=False,
    num_workers=workers,
    pin_memory=True,
    persistent_workers=workers > 0,
)

test_loader = DataLoader(
    test_ds,
    batch_size=256,
    shuffle=False,
    num_workers=workers,
    pin_memory=True,
    persistent_workers=workers > 0,
)

images, labels = next(iter(train_loader))

print()
print("DataLoader smoke test passed")
print("Image batch:", images.shape)
print("Label batch:", labels.shape)
print("Closed samples in batch:", int(labels.sum().item()))
print("Tensor range:", float(images.min()), "to", float(images.max()))
print("Workers:", workers)

Eye split verification passed

Train: 16940 images, 104 drivers
label
open      16134
closed      806

Validation: 3572 images, 23 drivers
label
open      3293
closed     279

Test: 3918 images, 23 drivers
label
open      3732
closed     186

DataLoader smoke test passed
Image batch: torch.Size([128, 3, 96, 192])
Label batch: torch.Size([128])
Closed samples in batch: 14
Tensor range: -2.1179039478302 to 2.640000104904175
Workers: 4


In [6]:
import math
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torchvision import models
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_recall_fscore_support,
    balanced_accuracy_score,
)

# --------------------------------------------------
# Reproducibility
# --------------------------------------------------

SEED = 20260829

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = True

device = torch.device("cuda:0")
use_amp = True

# --------------------------------------------------
# Model
# --------------------------------------------------

weights = models.ResNet18_Weights.DEFAULT

model = models.resnet18(weights=weights)

classifier_features = model.fc.in_features
model.fc = nn.Linear(classifier_features, 1)

model = model.to(device)
model = model.to(memory_format=torch.channels_last)

print("Device:", device)
print("Model: ResNet-18")
print("Input size: 96 × 192")
print("Using GPU:", torch.cuda.get_device_name(0))

# --------------------------------------------------
# Imbalance-aware loss
# --------------------------------------------------

train_counts = train_df["label"].value_counts()

number_open = int(train_counts["open"])
number_closed = int(train_counts["closed"])

raw_ratio = number_open / number_closed
positive_weight_value = math.sqrt(raw_ratio)

positive_weight = torch.tensor(
    [positive_weight_value],
    dtype=torch.float32,
    device=device,
)

criterion = nn.BCEWithLogitsLoss(
    pos_weight=positive_weight
)

print("Open samples:", number_open)
print("Closed samples:", number_closed)
print("Raw imbalance ratio:", round(raw_ratio, 4))
print("Applied positive weight:", round(positive_weight_value, 4))

# --------------------------------------------------
# Optimizer
# --------------------------------------------------

backbone_parameters = [
    parameter
    for name, parameter in model.named_parameters()
    if not name.startswith("fc.")
]

optimizer = torch.optim.AdamW(
    [
        {"params": backbone_parameters, "lr": 3e-5},
        {"params": model.fc.parameters(), "lr": 3e-4},
    ],
    weight_decay=1e-4,
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=1,
    min_lr=1e-6,
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=use_amp,
)

# --------------------------------------------------
# Validation
# --------------------------------------------------

def evaluate_eye_model(model, loader):
    model.eval()

    total_loss = 0.0
    all_targets = []
    all_probabilities = []

    with torch.inference_mode():
        for images, targets in loader:
            images = images.to(
                device,
                non_blocking=True,
                memory_format=torch.channels_last,
            )
            targets = targets.to(
                device,
                non_blocking=True,
            )

            with torch.amp.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=use_amp,
            ):
                logits = model(images).squeeze(1)
                loss = criterion(logits, targets)

            probabilities = torch.sigmoid(logits)

            total_loss += loss.item() * targets.size(0)
            all_targets.append(targets.cpu())
            all_probabilities.append(probabilities.cpu())

    targets = torch.cat(all_targets).numpy().astype(int)
    probabilities = torch.cat(all_probabilities).numpy()
    predictions = (probabilities >= 0.5).astype(int)

    precision, recall, f1, _ = precision_recall_fscore_support(
        targets,
        predictions,
        average="binary",
        zero_division=0,
    )

    metrics = {
        "loss": total_loss / len(loader.dataset),
        "ap": average_precision_score(targets, probabilities),
        "roc_auc": roc_auc_score(targets, probabilities),
        "balanced_accuracy_05": balanced_accuracy_score(
            targets,
            predictions,
        ),
        "closed_precision_05": precision,
        "closed_recall_05": recall,
        "closed_f1_05": f1,
    }

    return metrics

# --------------------------------------------------
# Training
# --------------------------------------------------

CHECKPOINT_PATH = OUTPUT_DIR / "drivealert_eye_resnet18_v1_best.pth"
HISTORY_PATH = OUTPUT_DIR / "drivealert_eye_resnet18_v1_history.csv"

MAX_EPOCHS = 15
EARLY_STOPPING_PATIENCE = 4

best_validation_ap = -1.0
epochs_without_improvement = 0
history = []

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()

    running_loss = 0.0
    processed_samples = 0

    for images, targets in train_loader:
        images = images.to(
            device,
            non_blocking=True,
            memory_format=torch.channels_last,
        )
        targets = targets.to(
            device,
            non_blocking=True,
        )

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=use_amp,
        ):
            logits = model(images).squeeze(1)
            loss = criterion(logits, targets)

        scaler.scale(loss).backward()

        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0,
        )

        scaler.step(optimizer)
        scaler.update()

        batch_size = targets.size(0)
        running_loss += loss.item() * batch_size
        processed_samples += batch_size

    train_loss = running_loss / processed_samples
    validation_metrics = evaluate_eye_model(model, val_loader)

    scheduler.step(validation_metrics["ap"])

    current_lr_backbone = optimizer.param_groups[0]["lr"]
    current_lr_classifier = optimizer.param_groups[1]["lr"]

    epoch_record = {
        "epoch": epoch,
        "train_loss": train_loss,
        **validation_metrics,
        "backbone_lr": current_lr_backbone,
        "classifier_lr": current_lr_classifier,
    }

    history.append(epoch_record)
    pd.DataFrame(history).to_csv(HISTORY_PATH, index=False)

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={train_loss:.4f} | "
        f"val_loss={validation_metrics['loss']:.4f} | "
        f"AP={validation_metrics['ap']:.4f} | "
        f"ROC-AUC={validation_metrics['roc_auc']:.4f} | "
        f"balanced_acc@0.5="
        f"{validation_metrics['balanced_accuracy_05']:.4f} | "
        f"closed_precision="
        f"{validation_metrics['closed_precision_05']:.4f} | "
        f"closed_recall="
        f"{validation_metrics['closed_recall_05']:.4f} | "
        f"closed_f1="
        f"{validation_metrics['closed_f1_05']:.4f} | "
        f"lr={current_lr_backbone:.2e}/"
        f"{current_lr_classifier:.2e}"
    )

    if validation_metrics["ap"] > best_validation_ap + 1e-4:
        best_validation_ap = validation_metrics["ap"]
        epochs_without_improvement = 0

        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "epoch": epoch,
                "validation_ap": best_validation_ap,
                "architecture": "resnet18",
                "input_height": 96,
                "input_width": 192,
                "label_mapping": EYE_LABELS,
                "positive_weight": positive_weight_value,
                "labels_are_ground_truth": False,
                "seed": SEED,
            },
            CHECKPOINT_PATH,
        )

        print("  Saved new best checkpoint.")

    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print("Early stopping triggered.")
        break

print()
print("Training complete")
print("Best validation AP:", round(best_validation_ap, 4))
print("Checkpoint:", CHECKPOINT_PATH)
print(
    "Checkpoint size:",
    round(CHECKPOINT_PATH.stat().st_size / 1024**2, 2),
    "MiB",
)
print("History:", HISTORY_PATH)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 140MB/s] 


Device: cuda:0
Model: ResNet-18
Input size: 96 × 192
Using GPU: Tesla T4
Open samples: 16134
Closed samples: 806
Raw imbalance ratio: 20.0174
Applied positive weight: 4.4741
Epoch 01 | train_loss=0.1660 | val_loss=0.2857 | AP=0.8493 | ROC-AUC=0.9746 | balanced_acc@0.5=0.8578 | closed_precision=0.8000 | closed_recall=0.7312 | closed_f1=0.7640 | lr=3.00e-05/3.00e-04
  Saved new best checkpoint.
Epoch 02 | train_loss=0.0491 | val_loss=0.2679 | AP=0.8844 | ROC-AUC=0.9798 | balanced_acc@0.5=0.8797 | closed_precision=0.8151 | closed_recall=0.7742 | closed_f1=0.7941 | lr=3.00e-05/3.00e-04
  Saved new best checkpoint.
Epoch 03 | train_loss=0.0317 | val_loss=0.3129 | AP=0.9081 | ROC-AUC=0.9825 | balanced_acc@0.5=0.8648 | closed_precision=0.9234 | closed_recall=0.7348 | closed_f1=0.8184 | lr=3.00e-05/3.00e-04
  Saved new best checkpoint.
Epoch 04 | train_loss=0.0225 | val_loss=0.2979 | AP=0.9251 | ROC-AUC=0.9858 | balanced_acc@0.5=0.8726 | closed_precision=0.9414 | closed_recall=0.7491 | closed_

In [7]:

import numpy as np
import pandas as pd
import torch

from sklearn.metrics import confusion_matrix

# --------------------------------------------------
# Load the best checkpoint
# --------------------------------------------------

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=device,
    weights_only=False,
)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("Loaded checkpoint epoch:", checkpoint["epoch"])
print("Saved validation AP:", checkpoint["validation_ap"])

# --------------------------------------------------
# Collect validation predictions
# --------------------------------------------------

validation_targets = []
validation_probabilities = []

with torch.inference_mode():
    for images, targets in val_loader:
        images = images.to(
            device,
            non_blocking=True,
            memory_format=torch.channels_last,
        )

        with torch.amp.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=use_amp,
        ):
            logits = model(images).squeeze(1)

        probabilities = torch.sigmoid(logits)

        validation_targets.append(targets.numpy())
        validation_probabilities.append(
            probabilities.cpu().numpy()
        )

validation_targets = np.concatenate(
    validation_targets
).astype(int)

validation_probabilities = np.concatenate(
    validation_probabilities
)

assert len(validation_targets) == len(val_df)

# Save predictions for reproducibility
validation_output = val_df[
    [
        "filepath",
        "label",
        "dataset",
        "person_id",
        "video_id",
        "frame_idx",
    ]
].copy()

validation_output["target"] = validation_targets
validation_output["closed_probability"] = validation_probabilities

VALIDATION_PREDICTIONS_PATH = (
    OUTPUT_DIR / "drivealert_eye_resnet18_v1_validation_predictions.csv"
)

validation_output.to_csv(
    VALIDATION_PREDICTIONS_PATH,
    index=False,
)

# --------------------------------------------------
# Calculate metrics across thresholds
# --------------------------------------------------

def calculate_threshold_metrics(threshold):
    predictions = (
        validation_probabilities >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        validation_targets,
        predictions,
        labels=[0, 1],
    ).ravel()

    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    specificity = tn / (tn + fp) if tn + fp else 0.0

    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall
        else 0.0
    )

    balanced_accuracy = (recall + specificity) / 2

    return {
        "threshold": float(threshold),
        "precision": precision,
        "closed_recall": recall,
        "open_specificity": specificity,
        "f1": f1,
        "balanced_accuracy": balanced_accuracy,
        "tp": int(tp),
        "fp": int(fp),
        "tn": int(tn),
        "fn": int(fn),
    }

thresholds = np.arange(0.01, 0.991, 0.005)

threshold_results = pd.DataFrame([
    calculate_threshold_metrics(threshold)
    for threshold in thresholds
])

def best_row(frame, sort_columns):
    return (
        frame.sort_values(
            sort_columns,
            ascending=[False] * len(sort_columns),
        )
        .iloc[0]
    )

candidates = {
    "Default threshold": calculate_threshold_metrics(0.5),

    "Best F1": best_row(
        threshold_results,
        ["f1", "closed_recall", "open_specificity"],
    ).to_dict(),

    "Best balanced accuracy": best_row(
        threshold_results,
        [
            "balanced_accuracy",
            "closed_recall",
            "open_specificity",
        ],
    ).to_dict(),
}

for required_specificity in (0.95, 0.97, 0.98):
    eligible = threshold_results[
        threshold_results["open_specificity"]
        >= required_specificity
    ]

    if not eligible.empty:
        candidates[
            f"Best recall with specificity >= "
            f"{required_specificity:.0%}"
        ] = best_row(
            eligible,
            ["closed_recall", "f1", "open_specificity"],
        ).to_dict()

high_recall = threshold_results[
    threshold_results["closed_recall"] >= 0.90
]

if not high_recall.empty:
    candidates[
        "Best specificity with recall >= 90%"
    ] = best_row(
        high_recall,
        ["open_specificity", "f1", "closed_recall"],
    ).to_dict()

# --------------------------------------------------
# Print candidates
# --------------------------------------------------

for name, metrics in candidates.items():
    print()
    print(name)
    print(f"  threshold:         {metrics['threshold']:.3f}")
    print(f"  precision:         {metrics['precision']:.4f}")
    print(f"  closed recall:     {metrics['closed_recall']:.4f}")
    print(f"  open specificity:  {metrics['open_specificity']:.4f}")
    print(f"  F1:                {metrics['f1']:.4f}")
    print(
        f"  balanced accuracy: "
        f"{metrics['balanced_accuracy']:.4f}"
    )
    print(
        f"  TP={int(metrics['tp'])} "
        f"FP={int(metrics['fp'])} "
        f"TN={int(metrics['tn'])} "
        f"FN={int(metrics['fn'])}"
    )

THRESHOLD_RESULTS_PATH = (
    OUTPUT_DIR / "drivealert_eye_resnet18_v1_threshold_results.csv"
)

threshold_results.to_csv(
    THRESHOLD_RESULTS_PATH,
    index=False,
)

print()
print("Validation predictions:", VALIDATION_PREDICTIONS_PATH)
print("Threshold results:", THRESHOLD_RESULTS_PATH)

Loaded checkpoint epoch: 12
Saved validation AP: 0.9393502121473437

Default threshold
  threshold:         0.500
  precision:         0.9492
  closed recall:     0.8029
  open specificity:  0.9964
  F1:                0.8699
  balanced accuracy: 0.8996
  TP=224 FP=12 TN=3281 FN=55

Best F1
  threshold:         0.375
  precision:         0.9419
  closed recall:     0.8136
  open specificity:  0.9957
  F1:                0.8731
  balanced accuracy: 0.9047
  TP=227 FP=14 TN=3279 FN=52

Best balanced accuracy
  threshold:         0.010
  precision:         0.7060
  closed recall:     0.9211
  open specificity:  0.9675
  F1:                0.7994
  balanced accuracy: 0.9443
  TP=257 FP=107 TN=3186 FN=22

Best recall with specificity >= 95%
  threshold:         0.010
  precision:         0.7060
  closed recall:     0.9211
  open specificity:  0.9675
  F1:                0.7994
  balanced accuracy: 0.9443
  TP=257 FP=107 TN=3186 FN=22

Best recall with specificity >= 97%
  threshold:        

In [8]:
import json
import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    confusion_matrix,
)

eligible_eye_thresholds = threshold_results[
    threshold_results["closed_recall"] >= 0.90
]

if eligible_eye_thresholds.empty:
    raise RuntimeError(
        "No validation threshold achieved 90% closed-eye recall. "
        "Do not evaluate the test set until the selection rule is revised."
    )

selected_eye_row = (
    eligible_eye_thresholds
    .sort_values(
        ["open_specificity", "f1", "closed_recall"],
        ascending=[False, False, False],
    )
    .iloc[0]
)

SELECTED_THRESHOLD = float(selected_eye_row["threshold"])

THRESHOLD_METADATA_PATH = (
    OUTPUT_DIR / "drivealert_eye_resnet18_v1_threshold_metadata.json"
)

threshold_metadata = {
    "model": "drivealert_eye_resnet18_v1",
    "architecture": "resnet18",
    "selected_threshold": SELECTED_THRESHOLD,
    "positive_class": "closed",
    "selected_using": "validation_split_only",
    "selection_rule": (
        "Highest open-eye specificity while maintaining "
        "at least 90% closed-eye recall"
    ),
    "validation_metrics": {
        "closed_recall": float(selected_eye_row["closed_recall"]),
        "open_specificity": float(selected_eye_row["open_specificity"]),
        "closed_precision": float(selected_eye_row["precision"]),
        "closed_f1": float(selected_eye_row["f1"]),
        "balanced_accuracy": float(
            selected_eye_row["balanced_accuracy"]
        ),
    },
    "labels_are_ground_truth": False,
    "warning": (
        "Threshold was selected using provisional weak labels. "
        "Independent human validation is required before production use."
    ),
}

THRESHOLD_METADATA_PATH.write_text(
    json.dumps(threshold_metadata, indent=2) + "\n"
)

print("Threshold locked before test evaluation:", SELECTED_THRESHOLD)

# --------------------------------------------------
# Collect untouched test predictions
# --------------------------------------------------

test_targets = []
test_probabilities = []

model.eval()

with torch.inference_mode():
    for images, targets in test_loader:
        images = images.to(
            device,
            non_blocking=True,
            memory_format=torch.channels_last,
        )

        with torch.amp.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=use_amp,
        ):
            logits = model(images).squeeze(1)

        probabilities = torch.sigmoid(logits)

        test_targets.append(targets.numpy())
        test_probabilities.append(
            probabilities.cpu().numpy()
        )

test_targets = np.concatenate(test_targets).astype(int)
test_probabilities = np.concatenate(test_probabilities)

assert len(test_targets) == len(test_df)

# --------------------------------------------------
# Fixed-threshold metrics
# --------------------------------------------------

def calculate_fixed_metrics(targets, probabilities, threshold):
    predictions = (probabilities >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        targets,
        predictions,
        labels=[0, 1],
    ).ravel()

    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    specificity = tn / (tn + fp) if tn + fp else 0.0

    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall
        else 0.0
    )

    return {
        "samples": int(len(targets)),
        "average_precision": float(
            average_precision_score(targets, probabilities)
        ),
        "roc_auc": float(
            roc_auc_score(targets, probabilities)
        ),
        "threshold": float(threshold),
        "balanced_accuracy": float(
            (recall + specificity) / 2
        ),
        "closed_precision": float(precision),
        "closed_recall": float(recall),
        "open_specificity": float(specificity),
        "closed_f1": float(f1),
        "tp": int(tp),
        "fp": int(fp),
        "tn": int(tn),
        "fn": int(fn),
    }

def print_metrics(name, metrics):
    print()
    print(name)
    print(f"  samples:           {metrics['samples']}")
    print(f"  threshold:         {metrics['threshold']:.3f}")
    print(
        f"  average precision: "
        f"{metrics['average_precision']:.4f}"
    )
    print(f"  ROC-AUC:           {metrics['roc_auc']:.4f}")
    print(
        f"  balanced accuracy: "
        f"{metrics['balanced_accuracy']:.4f}"
    )
    print(
        f"  closed precision:  "
        f"{metrics['closed_precision']:.4f}"
    )
    print(
        f"  closed recall:     "
        f"{metrics['closed_recall']:.4f}"
    )
    print(
        f"  open specificity:  "
        f"{metrics['open_specificity']:.4f}"
    )
    print(f"  closed F1:         {metrics['closed_f1']:.4f}")
    print(
        f"  TP={metrics['tp']} "
        f"FP={metrics['fp']} "
        f"TN={metrics['tn']} "
        f"FN={metrics['fn']}"
    )

complete_test_metrics = calculate_fixed_metrics(
    test_targets,
    test_probabilities,
    SELECTED_THRESHOLD,
)

print_metrics("Complete test set", complete_test_metrics)

# --------------------------------------------------
# Dataset-specific evaluation
# --------------------------------------------------

dataset_metrics = {}

for dataset_name in sorted(test_df["dataset"].unique()):
    mask = (
        test_df["dataset"].to_numpy()
        == dataset_name
    )

    metrics = calculate_fixed_metrics(
        test_targets[mask],
        test_probabilities[mask],
        SELECTED_THRESHOLD,
    )

    dataset_metrics[dataset_name] = metrics
    print_metrics(f"Dataset: {dataset_name}", metrics)

# --------------------------------------------------
# Save predictions and metrics
# --------------------------------------------------

test_output = test_df[
    [
        "filepath",
        "label",
        "dataset",
        "person_id",
        "video_id",
        "frame_idx",
    ]
].copy()

test_output["target"] = test_targets
test_output["closed_probability"] = test_probabilities
test_output["prediction"] = (
    test_probabilities >= SELECTED_THRESHOLD
).astype(int)

TEST_PREDICTIONS_PATH = (
    OUTPUT_DIR / "drivealert_eye_resnet18_v1_test_predictions.csv"
)

TEST_METRICS_PATH = (
    OUTPUT_DIR / "drivealert_eye_resnet18_v1_test_metrics.json"
)

test_output.to_csv(
    TEST_PREDICTIONS_PATH,
    index=False,
)

test_metrics_document = {
    "model": "drivealert_eye_resnet18_v1",
    "threshold_locked_before_test": True,
    "labels_are_ground_truth": False,
    "complete_test": complete_test_metrics,
    "by_dataset": dataset_metrics,
}

TEST_METRICS_PATH.write_text(
    json.dumps(test_metrics_document, indent=2) + "\n"
)

print()
print("Test predictions:", TEST_PREDICTIONS_PATH)
print("Test metrics:", TEST_METRICS_PATH)
print("Threshold metadata:", THRESHOLD_METADATA_PATH)

Threshold locked before test evaluation: 0.024999999999999998

Complete test set
  samples:           3918
  threshold:         0.025
  average precision: 0.9641
  ROC-AUC:           0.9962
  balanced accuracy: 0.9788
  closed precision:  0.6512
  closed recall:     0.9839
  open specificity:  0.9737
  closed F1:         0.7837
  TP=183 FP=98 TN=3634 FN=3

Dataset: uta_rldd
  samples:           2138
  threshold:         0.025
  average precision: 0.9779
  ROC-AUC:           0.9986
  balanced accuracy: 0.9811
  closed precision:  0.5575
  closed recall:     1.0000
  open specificity:  0.9623
  closed F1:         0.7159
  TP=97 FP=77 TN=1964 FN=0

Dataset: yawdd
  samples:           1780
  threshold:         0.025
  average precision: 0.9672
  ROC-AUC:           0.9958
  balanced accuracy: 0.9769
  closed precision:  0.8037
  closed recall:     0.9663
  open specificity:  0.9876
  closed F1:         0.8776
  TP=86 FP=21 TN=1670 FN=3

Test predictions: /kaggle/working/drivealert_eye_resne

In [9]:
import copy
import hashlib
import importlib.util
import json
import subprocess
import sys
import zipfile

import numpy as np
import torch
import torch.nn as nn
from torchvision import models

# --------------------------------------------------
# Ensure export dependencies are available
# --------------------------------------------------

missing_packages = []

if importlib.util.find_spec("onnx") is None:
    missing_packages.append("onnx")

if importlib.util.find_spec("onnxruntime") is None:
    missing_packages.append("onnxruntime")

if missing_packages:
    print("Installing:", missing_packages)
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        *missing_packages,
    ])

import onnx
import onnxruntime as ort

# --------------------------------------------------
# Paths
# --------------------------------------------------

ONNX_PATH = OUTPUT_DIR / "drivealert_eye_resnet18_v1.onnx"
MODEL_CARD_PATH = OUTPUT_DIR / "drivealert_eye_resnet18_v1_model_card.json"
BUNDLE_PATH = OUTPUT_DIR / "drivealert_eye_resnet18_v1_artifacts.zip"
CHECKSUM_PATH = OUTPUT_DIR / "drivealert_eye_resnet18_v1_artifacts.zip.sha256"

# --------------------------------------------------
# Reconstruct the best model on CPU
# --------------------------------------------------

best_checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False,
)

export_model = models.resnet18(weights=None)

classifier_features = export_model.fc.in_features
export_model.fc = nn.Linear(
    classifier_features,
    1,
)

export_model.load_state_dict(
    best_checkpoint["model_state_dict"]
)

export_model.eval()

dummy_input = torch.randn(
    1,
    3,
    96,
    192,
    dtype=torch.float32,
)

# --------------------------------------------------
# Export dynamic-batch ONNX model
# --------------------------------------------------

torch.onnx.export(
    export_model,
    dummy_input,
    str(ONNX_PATH),
    export_params=True,
    opset_version=17,
    do_constant_folding=True,
    input_names=["input"],
    output_names=["closed_logit"],
    dynamic_axes={
        "input": {0: "batch_size"},
        "closed_logit": {0: "batch_size"},
    },
    dynamo=False,
)

print("ONNX export complete:", ONNX_PATH)

# --------------------------------------------------
# Structural ONNX validation
# --------------------------------------------------

onnx_model = onnx.load(str(ONNX_PATH))
onnx.checker.check_model(onnx_model)

print("ONNX structural validation passed")

# --------------------------------------------------
# PyTorch/ONNX parity validation
# --------------------------------------------------

sample_images, _ = next(iter(test_loader))
sample_images = sample_images[:16].cpu().contiguous()

with torch.inference_mode():
    pytorch_logits = export_model(
        sample_images
    ).squeeze(1).numpy()

session = ort.InferenceSession(
    str(ONNX_PATH),
    providers=["CPUExecutionProvider"],
)

onnx_logits = session.run(
    ["closed_logit"],
    {
        "input": sample_images.numpy().astype(np.float32)
    },
)[0].reshape(-1)

maximum_logit_difference = float(
    np.max(np.abs(pytorch_logits - onnx_logits))
)

print(
    "Maximum PyTorch/ONNX logit difference:",
    maximum_logit_difference,
)

assert maximum_logit_difference < 1e-4, (
    "ONNX parity check failed"
)

print("PyTorch/ONNX parity validation passed")

# --------------------------------------------------
# Model card
# --------------------------------------------------

test_metrics = json.loads(
    TEST_METRICS_PATH.read_text()
)

model_card = {
    "model_name": "drivealert_eye_resnet18_v1",
    "task": "binary_eye_state_classification",
    "architecture": "resnet18",
    "positive_class": "closed",
    "output": {
        "name": "closed_logit",
        "activation": "sigmoid",
        "threshold": SELECTED_THRESHOLD,
    },
    "input": {
        "layout": "NCHW",
        "dtype": "float32",
        "height": 96,
        "width": 192,
        "color": "RGB",
        "normalization": {
            "mean": [0.485, 0.456, 0.406],
            "std": [0.229, 0.224, 0.225],
        },
    },
    "checkpoint_epoch": int(
        best_checkpoint["epoch"]
    ),
    "validation_ap": float(
        best_checkpoint["validation_ap"]
    ),
    "threshold_selection": {
        "split": "validation",
        "threshold": SELECTED_THRESHOLD,
        "rule": (
            "Highest specificity while preserving at least "
            "90% closed-eye recall"
        ),
    },
    "test_metrics": test_metrics,
    "training_dataset": (
        "krishnans2005/drivealert-processed-3"
    ),
    "participant_disjoint_splits": True,
    "labels_are_ground_truth": False,
    "production_status": "provisional_only",
    "warning": (
        "Training and evaluation labels are provisional weak labels. "
        "Independent human and real-driving validation are required."
    ),
}

MODEL_CARD_PATH.write_text(
    json.dumps(model_card, indent=2) + "\n"
)

print("Model card saved:", MODEL_CARD_PATH)

# --------------------------------------------------
# Bundle all eye-model artifacts
# --------------------------------------------------

artifact_paths = [
    CHECKPOINT_PATH,
    ONNX_PATH,
    MODEL_CARD_PATH,
    THRESHOLD_METADATA_PATH,
    TEST_METRICS_PATH,
    HISTORY_PATH,
    VALIDATION_PREDICTIONS_PATH,
    THRESHOLD_RESULTS_PATH,
    TEST_PREDICTIONS_PATH,
]

for path in artifact_paths:
    assert path.is_file(), f"Missing artifact: {path}"

with zipfile.ZipFile(
    BUNDLE_PATH,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as archive:
    for path in artifact_paths:
        archive.write(path, arcname=path.name)

with zipfile.ZipFile(BUNDLE_PATH, "r") as archive:
    broken_entry = archive.testzip()
    assert broken_entry is None, (
        f"ZIP validation failed: {broken_entry}"
    )

# --------------------------------------------------
# SHA-256 checksum
# --------------------------------------------------

digest = hashlib.sha256()

with BUNDLE_PATH.open("rb") as handle:
    for chunk in iter(
        lambda: handle.read(4 * 1024 * 1024),
        b"",
    ):
        digest.update(chunk)

bundle_sha256 = digest.hexdigest()

CHECKSUM_PATH.write_text(
    f"{bundle_sha256}  {BUNDLE_PATH.name}\n"
)

print()
print("Eye model export completed")
print(
    "Checkpoint:",
    CHECKPOINT_PATH.name,
    f"{CHECKPOINT_PATH.stat().st_size / 1024**2:.2f} MiB",
)
print(
    "ONNX:",
    ONNX_PATH.name,
    f"{ONNX_PATH.stat().st_size / 1024**2:.2f} MiB",
)
print(
    "Bundle:",
    BUNDLE_PATH.name,
    f"{BUNDLE_PATH.stat().st_size / 1024**2:.2f} MiB",
)
print("Bundle SHA-256:", bundle_sha256)
print("ZIP validation passed")

Installing: ['onnxruntime']
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 63.5 MB/s eta 0:00:00


/tmp/ipykernel_58/3226643663.py:85: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


ONNX export complete: /kaggle/working/drivealert_eye_resnet18_v1.onnx
ONNX structural validation passed
Maximum PyTorch/ONNX logit difference: 6.67572021484375e-06
PyTorch/ONNX parity validation passed
Model card saved: /kaggle/working/drivealert_eye_resnet18_v1_model_card.json

Eye model export completed
Checkpoint: drivealert_eye_resnet18_v1_best.pth 42.72 MiB
ONNX: drivealert_eye_resnet18_v1.onnx 42.63 MiB
Bundle: drivealert_eye_resnet18_v1_artifacts.zip 79.33 MiB
Bundle SHA-256: 72d9850dae55c4d8c9bb2154eaef3d24354fe9160cce51cb06ea88bff690f9f9
ZIP validation passed


In [10]:
from IPython.display import FileLink, display

display(
    FileLink(
        "/kaggle/working/drivealert_eye_resnet18_v1_artifacts.zip"
    )
)

display(
    FileLink(
        "/kaggle/working/drivealert_eye_resnet18_v1_artifacts.zip.sha256"
    )
)

/kaggle/working/drivealert_eye_resnet18_v1_artifacts.zip

/kaggle/working/drivealert_eye_resnet18_v1_artifacts.zip.sha256

In [11]:
from pathlib import Path
import os
import torch
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

MOUTH_LABELS = {
    "not_yawn": 0,
    "talking": 1,
    "yawn": 2,
}

def load_mouth_split(split):
    csv_path = (
        SPLIT_DIR / f"mouth_{split}_provisional.csv"
    )

    df = pd.read_csv(csv_path)

    required = {
        "filepath",
        "label",
        "person_id",
        "video_id",
        "dataset",
        "is_ground_truth",
    }

    missing_columns = required - set(df.columns)
    assert not missing_columns, (
        f"Missing columns: {missing_columns}"
    )

    assert set(df["label"]) <= set(MOUTH_LABELS)
    assert not df["filepath"].duplicated().any()

    df["image_path"] = df["filepath"].map(
        lambda path: DATASET_DIR / path
    )

    missing_files = [
        path
        for path in df["image_path"]
        if not path.is_file()
    ]

    assert not missing_files, (
        f"{len(missing_files)} mouth images are missing"
    )

    return df

mouth_train_df = load_mouth_split("train")
mouth_val_df = load_mouth_split("val")
mouth_test_df = load_mouth_split("test")

mouth_train_people = set(
    mouth_train_df["person_id"]
)
mouth_val_people = set(
    mouth_val_df["person_id"]
)
mouth_test_people = set(
    mouth_test_df["person_id"]
)

assert mouth_train_people.isdisjoint(
    mouth_val_people
)
assert mouth_train_people.isdisjoint(
    mouth_test_people
)
assert mouth_val_people.isdisjoint(
    mouth_test_people
)

print("Mouth split verification passed")
print()

print(
    "Train:",
    len(mouth_train_df),
    "images,",
    len(mouth_train_people),
    "drivers",
)
print(
    mouth_train_df["label"]
    .value_counts()
    .to_string()
)

print()
print(
    "Validation:",
    len(mouth_val_df),
    "images,",
    len(mouth_val_people),
    "drivers",
)
print(
    mouth_val_df["label"]
    .value_counts()
    .to_string()
)

print()
print(
    "Test:",
    len(mouth_test_df),
    "images,",
    len(mouth_test_people),
    "drivers",
)
print(
    mouth_test_df["label"]
    .value_counts()
    .to_string()
)

mouth_train_transform = transforms.Compose([
    transforms.Resize((96, 160), antialias=True),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomApply([
        transforms.ColorJitter(
            brightness=0.25,
            contrast=0.25,
            saturation=0.10,
        )
    ], p=0.7),
    transforms.RandomAffine(
        degrees=5,
        translate=(0.03, 0.03),
        scale=(0.95, 1.05),
        fill=(128, 128, 128),
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225),
    ),
])

mouth_eval_transform = transforms.Compose([
    transforms.Resize((96, 160), antialias=True),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225),
    ),
])

class MouthDataset(Dataset):
    def __init__(self, dataframe, transform):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]

        with Image.open(row["image_path"]) as image:
            image = image.convert("RGB")
            image = self.transform(image)

        label = torch.tensor(
            MOUTH_LABELS[row["label"]],
            dtype=torch.long,
        )

        return image, label

mouth_train_ds = MouthDataset(
    mouth_train_df,
    mouth_train_transform,
)

mouth_val_ds = MouthDataset(
    mouth_val_df,
    mouth_eval_transform,
)

mouth_test_ds = MouthDataset(
    mouth_test_df,
    mouth_eval_transform,
)

workers = min(4, os.cpu_count() or 2)

mouth_train_loader = DataLoader(
    mouth_train_ds,
    batch_size=128,
    shuffle=True,
    num_workers=workers,
    pin_memory=True,
    persistent_workers=workers > 0,
)

mouth_val_loader = DataLoader(
    mouth_val_ds,
    batch_size=256,
    shuffle=False,
    num_workers=workers,
    pin_memory=True,
    persistent_workers=workers > 0,
)

mouth_test_loader = DataLoader(
    mouth_test_ds,
    batch_size=256,
    shuffle=False,
    num_workers=workers,
    pin_memory=True,
    persistent_workers=workers > 0,
)

mouth_images, mouth_labels = next(
    iter(mouth_train_loader)
)

print()
print("Mouth DataLoader smoke test passed")
print("Image batch:", mouth_images.shape)
print("Label batch:", mouth_labels.shape)

batch_counts = torch.bincount(
    mouth_labels,
    minlength=len(MOUTH_LABELS),
)

for label_name, label_index in MOUTH_LABELS.items():
    print(
        f"{label_name} in batch:",
        int(batch_counts[label_index]),
    )

print(
    "Tensor range:",
    float(mouth_images.min()),
    "to",
    float(mouth_images.max()),
)
print("Workers:", workers)

Mouth split verification passed

Train: 12023 images, 62 drivers
label
not_yawn    6434
talking     4403
yawn        1186

Validation: 2884 images, 14 drivers
label
not_yawn    1475
talking     1105
yawn         304

Test: 2847 images, 14 drivers
label
not_yawn    1633
talking      935
yawn         279

Mouth DataLoader smoke test passed
Image batch: torch.Size([128, 3, 96, 160])
Label batch: torch.Size([128])
not_yawn in batch: 64
talking in batch: 52
yawn in batch: 12
Tensor range: -2.1179039478302 to 2.640000104904175
Workers: 4


In [12]:
import math
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torchvision import models
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_recall_fscore_support,
    balanced_accuracy_score,
)

# --------------------------------------------------
# Reproducibility and device
# --------------------------------------------------

SEED = 20260829

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = True

device = torch.device("cuda:0")
use_amp = True

# Release the previous GPU model; it is already exported.
if "model" in globals():
    del model

torch.cuda.empty_cache()

# --------------------------------------------------
# Mouth model
# --------------------------------------------------

pretrained_weights = (
    models.ResNet18_Weights.DEFAULT
)

mouth_model = models.resnet18(
    weights=pretrained_weights
)

classifier_features = mouth_model.fc.in_features

mouth_model.fc = nn.Linear(
    classifier_features,
    len(MOUTH_LABELS),
)

mouth_model = mouth_model.to(device)
mouth_model = mouth_model.to(
    memory_format=torch.channels_last
)

print("Device:", device)
print("Model: ResNet-18")
print("Classes:", MOUTH_LABELS)
print("Input size: 96 × 160")
print("Using GPU:", torch.cuda.get_device_name(0))

# --------------------------------------------------
# Conservative class balancing
# --------------------------------------------------

class_names = [
    "not_yawn",
    "talking",
    "yawn",
]

class_counts = np.array([
    int(
        mouth_train_df["label"]
        .value_counts()["not_yawn"]
    ),
    int(
        mouth_train_df["label"]
        .value_counts()["talking"]
    ),
    int(
        mouth_train_df["label"]
        .value_counts()["yawn"]
    ),
], dtype=np.float32)

# Square-root inverse frequency is less aggressive
# than full inverse-frequency weighting.
class_weights_array = np.sqrt(
    class_counts.sum()
    / (len(class_counts) * class_counts)
)

class_weights_array = (
    class_weights_array
    / class_weights_array.mean()
)

class_weights = torch.tensor(
    class_weights_array,
    dtype=torch.float32,
    device=device,
)

print("Class counts:", class_counts.astype(int).tolist())
print(
    "Class weights:",
    np.round(class_weights_array, 4).tolist(),
)

mouth_criterion = nn.CrossEntropyLoss(
    weight=class_weights,
    label_smoothing=0.03,
)

# --------------------------------------------------
# Optimizer and scheduler
# --------------------------------------------------

mouth_backbone_parameters = [
    parameter
    for name, parameter in mouth_model.named_parameters()
    if not name.startswith("fc.")
]

mouth_optimizer = torch.optim.AdamW(
    [
        {"params": mouth_backbone_parameters, "lr": 3e-5},
        {"params": mouth_model.fc.parameters(), "lr": 3e-4},
    ],
    weight_decay=1e-4,
)

mouth_scheduler = (
    torch.optim.lr_scheduler.ReduceLROnPlateau(
        mouth_optimizer,
        mode="max",
        factor=0.5,
        patience=1,
        min_lr=1e-6,
    )
)

mouth_scaler = torch.amp.GradScaler(
    "cuda",
    enabled=use_amp,
)

# --------------------------------------------------
# Validation
# --------------------------------------------------

def evaluate_mouth_model(model, loader):
    model.eval()

    total_loss = 0.0
    all_targets = []
    all_probabilities = []

    with torch.inference_mode():
        for images, targets in loader:
            images = images.to(
                device,
                non_blocking=True,
                memory_format=torch.channels_last,
            )
            targets = targets.to(
                device,
                non_blocking=True,
            )

            with torch.amp.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=use_amp,
            ):
                logits = model(images)
                loss = mouth_criterion(logits, targets)

            probabilities = torch.softmax(
                logits,
                dim=1,
            )

            total_loss += (
                loss.item() * targets.size(0)
            )

            all_targets.append(targets.cpu())
            all_probabilities.append(
                probabilities.cpu()
            )

    targets = torch.cat(all_targets).numpy()
    probabilities = torch.cat(
        all_probabilities
    ).numpy()

    predictions = probabilities.argmax(axis=1)

    one_hot_targets = np.eye(
        len(class_names)
    )[targets]

    per_class_ap = average_precision_score(
        one_hot_targets,
        probabilities,
        average=None,
    )

    macro_ap = float(np.mean(per_class_ap))

    macro_roc_auc = roc_auc_score(
        one_hot_targets,
        probabilities,
        average="macro",
        multi_class="ovr",
    )

    precision, recall, f1, _ = (
        precision_recall_fscore_support(
            targets,
            predictions,
            labels=list(range(len(class_names))),
            zero_division=0,
        )
    )

    metrics = {
        "loss": total_loss / len(loader.dataset),
        "macro_ap": macro_ap,
        "macro_roc_auc": float(macro_roc_auc),
        "balanced_accuracy": float(
            balanced_accuracy_score(
                targets,
                predictions,
            )
        ),
        "macro_f1": float(np.mean(f1)),
    }

    for index, class_name in enumerate(class_names):
        metrics[f"{class_name}_ap"] = float(
            per_class_ap[index]
        )
        metrics[f"{class_name}_precision"] = float(
            precision[index]
        )
        metrics[f"{class_name}_recall"] = float(
            recall[index]
        )
        metrics[f"{class_name}_f1"] = float(
            f1[index]
        )

    return metrics

# --------------------------------------------------
# Training
# --------------------------------------------------

MOUTH_CHECKPOINT_PATH = (
    OUTPUT_DIR / "drivealert_mouth_resnet18_v1_best.pth"
)

MOUTH_HISTORY_PATH = (
    OUTPUT_DIR / "drivealert_mouth_resnet18_v1_history.csv"
)

MAX_EPOCHS = 15
EARLY_STOPPING_PATIENCE = 4

best_macro_ap = -1.0
epochs_without_improvement = 0
mouth_history = []

for epoch in range(1, MAX_EPOCHS + 1):
    mouth_model.train()

    running_loss = 0.0
    processed_samples = 0

    for images, targets in mouth_train_loader:
        images = images.to(
            device,
            non_blocking=True,
            memory_format=torch.channels_last,
        )
        targets = targets.to(
            device,
            non_blocking=True,
        )

        mouth_optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=use_amp,
        ):
            logits = mouth_model(images)
            loss = mouth_criterion(
                logits,
                targets,
            )

        mouth_scaler.scale(loss).backward()

        mouth_scaler.unscale_(mouth_optimizer)

        torch.nn.utils.clip_grad_norm_(
            mouth_model.parameters(),
            max_norm=1.0,
        )

        mouth_scaler.step(mouth_optimizer)
        mouth_scaler.update()

        batch_size = targets.size(0)
        running_loss += loss.item() * batch_size
        processed_samples += batch_size

    train_loss = running_loss / processed_samples

    validation_metrics = evaluate_mouth_model(
        mouth_model,
        mouth_val_loader,
    )

    mouth_scheduler.step(
        validation_metrics["macro_ap"]
    )

    backbone_lr = (
        mouth_optimizer.param_groups[0]["lr"]
    )
    classifier_lr = (
        mouth_optimizer.param_groups[1]["lr"]
    )

    epoch_record = {
        "epoch": epoch,
        "train_loss": train_loss,
        **validation_metrics,
        "backbone_lr": backbone_lr,
        "classifier_lr": classifier_lr,
    }

    mouth_history.append(epoch_record)

    pd.DataFrame(mouth_history).to_csv(
        MOUTH_HISTORY_PATH,
        index=False,
    )

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={train_loss:.4f} | "
        f"val_loss={validation_metrics['loss']:.4f} | "
        f"macro_AP={validation_metrics['macro_ap']:.4f} | "
        f"macro_AUC={validation_metrics['macro_roc_auc']:.4f} | "
        f"balanced_acc={validation_metrics['balanced_accuracy']:.4f} | "
        f"macro_F1={validation_metrics['macro_f1']:.4f} | "
        f"yawn_AP={validation_metrics['yawn_ap']:.4f} | "
        f"yawn_precision="
        f"{validation_metrics['yawn_precision']:.4f} | "
        f"yawn_recall="
        f"{validation_metrics['yawn_recall']:.4f} | "
        f"yawn_F1="
        f"{validation_metrics['yawn_f1']:.4f} | "
        f"lr={backbone_lr:.2e}/{classifier_lr:.2e}"
    )

    if (
        validation_metrics["macro_ap"]
        > best_macro_ap + 1e-4
    ):
        best_macro_ap = (
            validation_metrics["macro_ap"]
        )
        epochs_without_improvement = 0

        torch.save(
            {
                "model_state_dict": (
                    mouth_model.state_dict()
                ),
                "epoch": epoch,
                "validation_macro_ap": (
                    best_macro_ap
                ),
                "architecture": (
                    "resnet18"
                ),
                "input_height": 96,
                "input_width": 160,
                "label_mapping": MOUTH_LABELS,
                "class_weights": (
                    class_weights_array.tolist()
                ),
                "labels_are_ground_truth": False,
                "seed": SEED,
            },
            MOUTH_CHECKPOINT_PATH,
        )

        print("  Saved new best checkpoint.")

    else:
        epochs_without_improvement += 1

    if (
        epochs_without_improvement
        >= EARLY_STOPPING_PATIENCE
    ):
        print("Early stopping triggered.")
        break

print()
print("Mouth training complete")
print(
    "Best validation macro AP:",
    round(best_macro_ap, 4),
)
print("Checkpoint:", MOUTH_CHECKPOINT_PATH)
print(
    "Checkpoint size:",
    round(
        MOUTH_CHECKPOINT_PATH.stat().st_size
        / 1024**2,
        2,
    ),
    "MiB",
)
print("History:", MOUTH_HISTORY_PATH)

Device: cuda:0
Model: ResNet-18
Classes: {'not_yawn': 0, 'talking': 1, 'yawn': 2}
Input size: 96 × 160
Using GPU: Tesla T4
Class counts: [6434, 4403, 1186]
Class weights: [0.6610999703407288, 0.7990999817848206, 1.5398000478744507]
Epoch 01 | train_loss=0.4107 | val_loss=0.3721 | macro_AP=0.9496 | macro_AUC=0.9711 | balanced_acc=0.9022 | macro_F1=0.8879 | yawn_AP=0.9292 | yawn_precision=0.7736 | yawn_recall=0.8882 | yawn_F1=0.8270 | lr=3.00e-05/3.00e-04
  Saved new best checkpoint.
Epoch 02 | train_loss=0.2579 | val_loss=0.4022 | macro_AP=0.9447 | macro_AUC=0.9685 | balanced_acc=0.8915 | macro_F1=0.8556 | yawn_AP=0.9253 | yawn_precision=0.6395 | yawn_recall=0.9046 | yawn_F1=0.7493 | lr=3.00e-05/3.00e-04
Epoch 03 | train_loss=0.2317 | val_loss=0.3547 | macro_AP=0.9496 | macro_AUC=0.9707 | balanced_acc=0.9191 | macro_F1=0.9070 | yawn_AP=0.9273 | yawn_precision=0.8047 | yawn_recall=0.9079 | yawn_F1=0.8532 | lr=1.50e-05/1.50e-04
Epoch 04 | train_loss=0.2144 | val_loss=0.3357 | macro_AP=0.9

In [13]:
import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    confusion_matrix,
    average_precision_score,
    roc_auc_score,
    precision_recall_fscore_support,
)

# --------------------------------------------------
# Load the best mouth checkpoint
# --------------------------------------------------

mouth_checkpoint = torch.load(
    MOUTH_CHECKPOINT_PATH,
    map_location=device,
    weights_only=False,
)

mouth_model.load_state_dict(
    mouth_checkpoint["model_state_dict"]
)

mouth_model.eval()

print(
    "Loaded checkpoint epoch:",
    mouth_checkpoint["epoch"],
)
print(
    "Saved validation macro AP:",
    mouth_checkpoint["validation_macro_ap"],
)

# --------------------------------------------------
# Collect validation predictions
# --------------------------------------------------

mouth_val_targets = []
mouth_val_probabilities = []

with torch.inference_mode():
    for images, targets in mouth_val_loader:
        images = images.to(
            device,
            non_blocking=True,
            memory_format=torch.channels_last,
        )

        with torch.amp.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=use_amp,
        ):
            logits = mouth_model(images)

        probabilities = torch.softmax(
            logits,
            dim=1,
        )

        mouth_val_targets.append(targets.numpy())
        mouth_val_probabilities.append(
            probabilities.cpu().numpy()
        )

mouth_val_targets = np.concatenate(
    mouth_val_targets
).astype(int)

mouth_val_probabilities = np.concatenate(
    mouth_val_probabilities
)

assert len(mouth_val_targets) == len(mouth_val_df)

argmax_predictions = mouth_val_probabilities.argmax(
    axis=1
)

# --------------------------------------------------
# Three-class validation report
# --------------------------------------------------

matrix = confusion_matrix(
    mouth_val_targets,
    argmax_predictions,
    labels=[0, 1, 2],
)

matrix_df = pd.DataFrame(
    matrix,
    index=[
        "true_not_yawn",
        "true_talking",
        "true_yawn",
    ],
    columns=[
        "pred_not_yawn",
        "pred_talking",
        "pred_yawn",
    ],
)

print()
print("Three-class confusion matrix")
print(matrix_df.to_string())

precision, recall, f1, support = (
    precision_recall_fscore_support(
        mouth_val_targets,
        argmax_predictions,
        labels=[0, 1, 2],
        zero_division=0,
    )
)

print()
print("Argmax per-class metrics")

for index, class_name in enumerate(class_names):
    print(
        f"  {class_name:10s} | "
        f"precision={precision[index]:.4f} | "
        f"recall={recall[index]:.4f} | "
        f"F1={f1[index]:.4f} | "
        f"support={int(support[index])}"
    )

# --------------------------------------------------
# Yawn-vs-rest threshold analysis
# --------------------------------------------------

YAWN_INDEX = MOUTH_LABELS["yawn"]

yawn_targets = (
    mouth_val_targets == YAWN_INDEX
).astype(int)

yawn_probabilities = mouth_val_probabilities[
    :,
    YAWN_INDEX,
]

print()
print(
    "Yawn validation AP:",
    round(
        average_precision_score(
            yawn_targets,
            yawn_probabilities,
        ),
        4,
    ),
)

print(
    "Yawn validation ROC-AUC:",
    round(
        roc_auc_score(
            yawn_targets,
            yawn_probabilities,
        ),
        4,
    ),
)

def calculate_yawn_metrics(threshold):
    predictions = (
        yawn_probabilities >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        yawn_targets,
        predictions,
        labels=[0, 1],
    ).ravel()

    precision_value = (
        tp / (tp + fp)
        if tp + fp
        else 0.0
    )

    recall_value = (
        tp / (tp + fn)
        if tp + fn
        else 0.0
    )

    specificity_value = (
        tn / (tn + fp)
        if tn + fp
        else 0.0
    )

    f1_value = (
        2
        * precision_value
        * recall_value
        / (precision_value + recall_value)
        if precision_value + recall_value
        else 0.0
    )

    return {
        "threshold": float(threshold),
        "precision": precision_value,
        "yawn_recall": recall_value,
        "non_yawn_specificity": specificity_value,
        "f1": f1_value,
        "balanced_accuracy": (
            recall_value + specificity_value
        ) / 2,
        "tp": int(tp),
        "fp": int(fp),
        "tn": int(tn),
        "fn": int(fn),
    }

thresholds = np.arange(
    0.01,
    0.991,
    0.005,
)

mouth_threshold_results = pd.DataFrame([
    calculate_yawn_metrics(threshold)
    for threshold in thresholds
])

def select_best(frame, columns):
    return (
        frame.sort_values(
            columns,
            ascending=[False] * len(columns),
        )
        .iloc[0]
    )

candidate_metrics = {
    "Probability threshold 0.5":
        calculate_yawn_metrics(0.5),

    "Best yawn F1":
        select_best(
            mouth_threshold_results,
            [
                "f1",
                "yawn_recall",
                "non_yawn_specificity",
            ],
        ).to_dict(),

    "Best balanced accuracy":
        select_best(
            mouth_threshold_results,
            [
                "balanced_accuracy",
                "yawn_recall",
                "non_yawn_specificity",
            ],
        ).to_dict(),
}

for required_recall in (0.90, 0.95):
    eligible = mouth_threshold_results[
        mouth_threshold_results["yawn_recall"]
        >= required_recall
    ]

    if not eligible.empty:
        candidate_metrics[
            f"Best specificity with recall >= "
            f"{required_recall:.0%}"
        ] = select_best(
            eligible,
            [
                "non_yawn_specificity",
                "f1",
                "yawn_recall",
            ],
        ).to_dict()

for name, metrics in candidate_metrics.items():
    print()
    print(name)
    print(
        f"  threshold:           "
        f"{metrics['threshold']:.3f}"
    )
    print(
        f"  precision:           "
        f"{metrics['precision']:.4f}"
    )
    print(
        f"  yawn recall:         "
        f"{metrics['yawn_recall']:.4f}"
    )
    print(
        f"  non-yawn specificity:"
        f"{metrics['non_yawn_specificity']:.4f}"
    )
    print(
        f"  F1:                  "
        f"{metrics['f1']:.4f}"
    )
    print(
        f"  balanced accuracy:   "
        f"{metrics['balanced_accuracy']:.4f}"
    )
    print(
        f"  TP={int(metrics['tp'])} "
        f"FP={int(metrics['fp'])} "
        f"TN={int(metrics['tn'])} "
        f"FN={int(metrics['fn'])}"
    )

# --------------------------------------------------
# Save validation analysis
# --------------------------------------------------

MOUTH_VALIDATION_PREDICTIONS_PATH = (
    OUTPUT_DIR
    / "drivealert_mouth_resnet18_v1_validation_predictions.csv"
)

MOUTH_THRESHOLD_RESULTS_PATH = (
    OUTPUT_DIR
    / "drivealert_mouth_resnet18_v1_threshold_results.csv"
)

mouth_validation_output = mouth_val_df[
    [
        "filepath",
        "label",
        "dataset",
        "person_id",
        "video_id",
        "frame_idx",
    ]
].copy()

mouth_validation_output["target"] = (
    mouth_val_targets
)

mouth_validation_output["predicted_class"] = (
    argmax_predictions
)

mouth_validation_output["not_yawn_probability"] = (
    mouth_val_probabilities[:, 0]
)

mouth_validation_output["talking_probability"] = (
    mouth_val_probabilities[:, 1]
)

mouth_validation_output["yawn_probability"] = (
    mouth_val_probabilities[:, 2]
)

mouth_validation_output.to_csv(
    MOUTH_VALIDATION_PREDICTIONS_PATH,
    index=False,
)

mouth_threshold_results.to_csv(
    MOUTH_THRESHOLD_RESULTS_PATH,
    index=False,
)

print()
print(
    "Validation predictions:",
    MOUTH_VALIDATION_PREDICTIONS_PATH,
)
print(
    "Threshold results:",
    MOUTH_THRESHOLD_RESULTS_PATH,
)

Loaded checkpoint epoch: 4
Saved validation macro AP: 0.954726705291404

Three-class confusion matrix
               pred_not_yawn  pred_talking  pred_yawn
true_not_yawn           1414            56          5
true_talking              54          1002         49
true_yawn                  2            26        276

Argmax per-class metrics
  not_yawn   | precision=0.9619 | recall=0.9586 | F1=0.9603 | support=1475
  talking    | precision=0.9244 | recall=0.9068 | F1=0.9155 | support=1105
  yawn       | precision=0.8364 | recall=0.9079 | F1=0.8707 | support=304

Yawn validation AP: 0.9364
Yawn validation ROC-AUC: 0.9633

Probability threshold 0.5
  threshold:           0.500
  precision:           0.8466
  yawn recall:         0.9079
  non-yawn specificity:0.9806
  F1:                  0.8762
  balanced accuracy:   0.9443
  TP=276 FP=50 TN=2530 FN=28

Best yawn F1
  threshold:           0.835
  precision:           0.9745
  yawn recall:         0.8816
  non-yawn specificity:0.9973
  F1

In [14]:
import json
import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    confusion_matrix,
    average_precision_score,
    roc_auc_score,
    precision_recall_fscore_support,
    balanced_accuracy_score,
)

eligible_yawn_thresholds = mouth_threshold_results[
    mouth_threshold_results["yawn_recall"] >= 0.90
]

if eligible_yawn_thresholds.empty:
    raise RuntimeError(
        "No validation threshold achieved 90% yawn recall. "
        "Do not evaluate the test set until the selection rule is revised."
    )

selected_yawn_row = (
    eligible_yawn_thresholds
    .sort_values(
        ["non_yawn_specificity", "f1", "yawn_recall"],
        ascending=[False, False, False],
    )
    .iloc[0]
)

SELECTED_YAWN_THRESHOLD = float(selected_yawn_row["threshold"])

MOUTH_THRESHOLD_METADATA_PATH = (
    OUTPUT_DIR
    / "drivealert_mouth_resnet18_v1_threshold_metadata.json"
)

mouth_threshold_metadata = {
    "model": "drivealert_mouth_resnet18_v1",
    "architecture": "resnet18",
    "selected_yawn_threshold": (
        SELECTED_YAWN_THRESHOLD
    ),
    "selected_using": "validation_split_only",
    "selection_rule": (
        "Highest non-yawn specificity while maintaining "
        "at least 90% yawn recall"
    ),
    "decision_policy": (
        f"Predict yawn when yawn probability >= "
        f"{SELECTED_YAWN_THRESHOLD:.3f}. Otherwise choose the higher "
        "probability between not_yawn and talking."
    ),
    "validation_metrics": {
        "yawn_precision": float(selected_yawn_row["precision"]),
        "yawn_recall": float(selected_yawn_row["yawn_recall"]),
        "non_yawn_specificity": float(
            selected_yawn_row["non_yawn_specificity"]
        ),
        "yawn_f1": float(selected_yawn_row["f1"]),
        "balanced_accuracy": float(
            selected_yawn_row["balanced_accuracy"]
        ),
    },
    "labels_are_ground_truth": False,
    "warning": (
        "Threshold was selected using provisional weak labels. "
        "Independent human validation is required before "
        "production use."
    ),
}

MOUTH_THRESHOLD_METADATA_PATH.write_text(
    json.dumps(
        mouth_threshold_metadata,
        indent=2,
    ) + "\n"
)

print(
    "Yawn threshold locked before test evaluation:",
    SELECTED_YAWN_THRESHOLD,
)

# --------------------------------------------------
# Collect untouched test predictions
# --------------------------------------------------

mouth_test_targets = []
mouth_test_probabilities = []

mouth_model.eval()

with torch.inference_mode():
    for images, targets in mouth_test_loader:
        images = images.to(
            device,
            non_blocking=True,
            memory_format=torch.channels_last,
        )

        with torch.amp.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=use_amp,
        ):
            logits = mouth_model(images)

        probabilities = torch.softmax(
            logits,
            dim=1,
        )

        mouth_test_targets.append(targets.numpy())
        mouth_test_probabilities.append(
            probabilities.cpu().numpy()
        )

mouth_test_targets = np.concatenate(
    mouth_test_targets
).astype(int)

mouth_test_probabilities = np.concatenate(
    mouth_test_probabilities
)

assert len(mouth_test_targets) == len(
    mouth_test_df
)

# --------------------------------------------------
# Apply locked inference policy
# --------------------------------------------------

# First choose between not_yawn and talking.
mouth_test_predictions = (
    mouth_test_probabilities[:, :2]
    .argmax(axis=1)
)

# Override with yawn when the locked threshold passes.
mouth_test_predictions[
    mouth_test_probabilities[:, 2]
    >= SELECTED_YAWN_THRESHOLD
] = MOUTH_LABELS["yawn"]

# --------------------------------------------------
# Three-class metrics
# --------------------------------------------------

mouth_test_matrix = confusion_matrix(
    mouth_test_targets,
    mouth_test_predictions,
    labels=[0, 1, 2],
)

mouth_test_matrix_df = pd.DataFrame(
    mouth_test_matrix,
    index=[
        "true_not_yawn",
        "true_talking",
        "true_yawn",
    ],
    columns=[
        "pred_not_yawn",
        "pred_talking",
        "pred_yawn",
    ],
)

precision, recall, f1, support = (
    precision_recall_fscore_support(
        mouth_test_targets,
        mouth_test_predictions,
        labels=[0, 1, 2],
        zero_division=0,
    )
)

one_hot_test_targets = np.eye(
    len(class_names)
)[mouth_test_targets]

per_class_ap = average_precision_score(
    one_hot_test_targets,
    mouth_test_probabilities,
    average=None,
)

macro_ap = float(np.mean(per_class_ap))

macro_roc_auc = float(
    roc_auc_score(
        one_hot_test_targets,
        mouth_test_probabilities,
        average="macro",
        multi_class="ovr",
    )
)

macro_f1 = float(np.mean(f1))

multiclass_balanced_accuracy = float(
    balanced_accuracy_score(
        mouth_test_targets,
        mouth_test_predictions,
    )
)

print()
print("Mouth test confusion matrix")
print(mouth_test_matrix_df.to_string())

print()
print("Test per-class metrics")

per_class_metrics = {}

for index, class_name in enumerate(class_names):
    per_class_metrics[class_name] = {
        "average_precision": float(
            per_class_ap[index]
        ),
        "precision": float(precision[index]),
        "recall": float(recall[index]),
        "f1": float(f1[index]),
        "support": int(support[index]),
    }

    print(
        f"  {class_name:10s} | "
        f"AP={per_class_ap[index]:.4f} | "
        f"precision={precision[index]:.4f} | "
        f"recall={recall[index]:.4f} | "
        f"F1={f1[index]:.4f} | "
        f"support={int(support[index])}"
    )

print()
print("Overall mouth test metrics")
print(f"  macro AP:          {macro_ap:.4f}")
print(f"  macro ROC-AUC:     {macro_roc_auc:.4f}")
print(f"  macro F1:          {macro_f1:.4f}")
print(
    f"  balanced accuracy: "
    f"{multiclass_balanced_accuracy:.4f}"
)

# --------------------------------------------------
# Yawn-vs-rest fixed-threshold metrics
# --------------------------------------------------

yawn_test_targets = (
    mouth_test_targets == MOUTH_LABELS["yawn"]
).astype(int)

yawn_test_predictions = (
    mouth_test_probabilities[:, 2]
    >= SELECTED_YAWN_THRESHOLD
).astype(int)

tn, fp, fn, tp = confusion_matrix(
    yawn_test_targets,
    yawn_test_predictions,
    labels=[0, 1],
).ravel()

yawn_precision = (
    tp / (tp + fp)
    if tp + fp
    else 0.0
)

yawn_recall = (
    tp / (tp + fn)
    if tp + fn
    else 0.0
)

non_yawn_specificity = (
    tn / (tn + fp)
    if tn + fp
    else 0.0
)

yawn_f1 = (
    2
    * yawn_precision
    * yawn_recall
    / (yawn_precision + yawn_recall)
    if yawn_precision + yawn_recall
    else 0.0
)

yawn_balanced_accuracy = (
    yawn_recall + non_yawn_specificity
) / 2

print()
print("Locked yawn-threshold test metrics")
print(
    f"  threshold:            "
    f"{SELECTED_YAWN_THRESHOLD:.3f}"
)
print(
    f"  yawn precision:       "
    f"{yawn_precision:.4f}"
)
print(
    f"  yawn recall:          "
    f"{yawn_recall:.4f}"
)
print(
    f"  non-yawn specificity: "
    f"{non_yawn_specificity:.4f}"
)
print(f"  yawn F1:              {yawn_f1:.4f}")
print(
    f"  balanced accuracy:    "
    f"{yawn_balanced_accuracy:.4f}"
)
print(f"  TP={tp} FP={fp} TN={tn} FN={fn}")

# --------------------------------------------------
# Save predictions and metrics
# --------------------------------------------------

MOUTH_TEST_PREDICTIONS_PATH = (
    OUTPUT_DIR
    / "drivealert_mouth_resnet18_v1_test_predictions.csv"
)

MOUTH_TEST_METRICS_PATH = (
    OUTPUT_DIR
    / "drivealert_mouth_resnet18_v1_test_metrics.json"
)

mouth_test_output = mouth_test_df[
    [
        "filepath",
        "label",
        "dataset",
        "person_id",
        "video_id",
        "frame_idx",
    ]
].copy()

mouth_test_output["target"] = mouth_test_targets
mouth_test_output["prediction"] = (
    mouth_test_predictions
)

mouth_test_output["not_yawn_probability"] = (
    mouth_test_probabilities[:, 0]
)

mouth_test_output["talking_probability"] = (
    mouth_test_probabilities[:, 1]
)

mouth_test_output["yawn_probability"] = (
    mouth_test_probabilities[:, 2]
)

mouth_test_output.to_csv(
    MOUTH_TEST_PREDICTIONS_PATH,
    index=False,
)

mouth_test_metrics_document = {
    "model": "drivealert_mouth_resnet18_v1",
    "threshold_locked_before_test": True,
    "selected_yawn_threshold": (
        SELECTED_YAWN_THRESHOLD
    ),
    "labels_are_ground_truth": False,
    "macro_average_precision": macro_ap,
    "macro_roc_auc": macro_roc_auc,
    "macro_f1": macro_f1,
    "multiclass_balanced_accuracy": (
        multiclass_balanced_accuracy
    ),
    "per_class": per_class_metrics,
    "confusion_matrix": (
        mouth_test_matrix.tolist()
    ),
    "yawn_vs_rest": {
        "precision": float(yawn_precision),
        "recall": float(yawn_recall),
        "specificity": float(
            non_yawn_specificity
        ),
        "f1": float(yawn_f1),
        "balanced_accuracy": float(
            yawn_balanced_accuracy
        ),
        "tp": int(tp),
        "fp": int(fp),
        "tn": int(tn),
        "fn": int(fn),
    },
}

MOUTH_TEST_METRICS_PATH.write_text(
    json.dumps(
        mouth_test_metrics_document,
        indent=2,
    ) + "\n"
)

print()
print(
    "Test predictions:",
    MOUTH_TEST_PREDICTIONS_PATH,
)
print(
    "Test metrics:",
    MOUTH_TEST_METRICS_PATH,
)
print(
    "Threshold metadata:",
    MOUTH_THRESHOLD_METADATA_PATH,
)

Yawn threshold locked before test evaluation: 0.6499999999999999

Mouth test confusion matrix
               pred_not_yawn  pred_talking  pred_yawn
true_not_yawn           1584            49          0
true_talking              33           899          3
true_yawn                  0            18        261

Test per-class metrics
  not_yawn   | AP=0.9939 | precision=0.9796 | recall=0.9700 | F1=0.9748 | support=1633
  talking    | AP=0.9802 | precision=0.9306 | recall=0.9615 | F1=0.9458 | support=935
  yawn       | AP=0.9871 | precision=0.9886 | recall=0.9355 | F1=0.9613 | support=279

Overall mouth test metrics
  macro AP:          0.9870
  macro ROC-AUC:     0.9940
  macro F1:          0.9606
  balanced accuracy: 0.9557

Locked yawn-threshold test metrics
  threshold:            0.650
  yawn precision:       0.9886
  yawn recall:          0.9355
  non-yawn specificity: 0.9988
  yawn F1:              0.9613
  balanced accuracy:    0.9672
  TP=261 FP=3 TN=2565 FN=18

Test predictions:

In [15]:
import hashlib
import json
import zipfile

import numpy as np
import torch
import torch.nn as nn
import onnx
import onnxruntime as ort

from torchvision import models

# --------------------------------------------------
# Artifact paths
# --------------------------------------------------

MOUTH_ONNX_PATH = (
    OUTPUT_DIR / "drivealert_mouth_resnet18_v1.onnx"
)

MOUTH_MODEL_CARD_PATH = (
    OUTPUT_DIR / "drivealert_mouth_resnet18_v1_model_card.json"
)

MOUTH_BUNDLE_PATH = (
    OUTPUT_DIR / "drivealert_mouth_resnet18_v1_artifacts.zip"
)

MOUTH_CHECKSUM_PATH = (
    OUTPUT_DIR
    / "drivealert_mouth_resnet18_v1_artifacts.zip.sha256"
)

# --------------------------------------------------
# Reconstruct best model on CPU
# --------------------------------------------------

best_mouth_checkpoint = torch.load(
    MOUTH_CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False,
)

mouth_export_model = (
    models.resnet18(weights=None)
)

classifier_features = mouth_export_model.fc.in_features

mouth_export_model.fc = nn.Linear(
    classifier_features,
    len(MOUTH_LABELS),
)

mouth_export_model.load_state_dict(
    best_mouth_checkpoint["model_state_dict"]
)

mouth_export_model.eval()

mouth_dummy_input = torch.randn(
    1,
    3,
    96,
    160,
    dtype=torch.float32,
)

# --------------------------------------------------
# ONNX export
# --------------------------------------------------

torch.onnx.export(
    mouth_export_model,
    mouth_dummy_input,
    str(MOUTH_ONNX_PATH),
    export_params=True,
    opset_version=17,
    do_constant_folding=True,
    input_names=["input"],
    output_names=["class_logits"],
    dynamic_axes={
        "input": {0: "batch_size"},
        "class_logits": {0: "batch_size"},
    },
    dynamo=False,
)

print(
    "ONNX export complete:",
    MOUTH_ONNX_PATH,
)

# --------------------------------------------------
# Structural validation
# --------------------------------------------------

mouth_onnx_model = onnx.load(
    str(MOUTH_ONNX_PATH)
)

onnx.checker.check_model(
    mouth_onnx_model
)

print("ONNX structural validation passed")

# --------------------------------------------------
# PyTorch/ONNX parity validation
# --------------------------------------------------

mouth_sample_images, _ = next(
    iter(mouth_test_loader)
)

mouth_sample_images = (
    mouth_sample_images[:16]
    .cpu()
    .contiguous()
)

with torch.inference_mode():
    pytorch_mouth_logits = (
        mouth_export_model(
            mouth_sample_images
        ).numpy()
    )

mouth_session = ort.InferenceSession(
    str(MOUTH_ONNX_PATH),
    providers=["CPUExecutionProvider"],
)

onnx_mouth_logits = mouth_session.run(
    ["class_logits"],
    {
        "input": (
            mouth_sample_images
            .numpy()
            .astype(np.float32)
        )
    },
)[0]

maximum_mouth_logit_difference = float(
    np.max(
        np.abs(
            pytorch_mouth_logits
            - onnx_mouth_logits
        )
    )
)

print(
    "Maximum PyTorch/ONNX logit difference:",
    maximum_mouth_logit_difference,
)

assert maximum_mouth_logit_difference < 2e-4, (
    "Mouth ONNX parity check failed"
)

print("PyTorch/ONNX parity validation passed")

# --------------------------------------------------
# Model card
# --------------------------------------------------

mouth_test_metrics = json.loads(
    MOUTH_TEST_METRICS_PATH.read_text()
)

mouth_model_card = {
    "model_name": "drivealert_mouth_resnet18_v1",
    "task": "three_class_mouth_event_classification",
    "architecture": "resnet18",
    "classes": MOUTH_LABELS,
    "output": {
        "name": "class_logits",
        "activation": "softmax",
    },
    "decision_policy": {
        "yawn_threshold": (
            SELECTED_YAWN_THRESHOLD
        ),
        "rule": (
            "Predict yawn when yawn probability "
            f"is at least {SELECTED_YAWN_THRESHOLD:.3f}. Otherwise select "
            "the larger probability between "
            "not_yawn and talking."
        ),
    },
    "input": {
        "layout": "NCHW",
        "dtype": "float32",
        "height": 96,
        "width": 160,
        "color": "RGB",
        "normalization": {
            "mean": [0.485, 0.456, 0.406],
            "std": [0.229, 0.224, 0.225],
        },
    },
    "checkpoint_epoch": int(
        best_mouth_checkpoint["epoch"]
    ),
    "validation_macro_ap": float(
        best_mouth_checkpoint[
            "validation_macro_ap"
        ]
    ),
    "threshold_selection": {
        "split": "validation",
        "yawn_threshold": (
            SELECTED_YAWN_THRESHOLD
        ),
        "rule": (
            "Highest non-yawn specificity while "
            "maintaining at least 90% yawn recall"
        ),
    },
    "test_metrics": mouth_test_metrics,
    "training_dataset": (
        "krishnans2005/drivealert-processed-3"
    ),
    "participant_disjoint_splits": True,
    "labels_are_ground_truth": False,
    "production_status": "provisional_only",
    "warning": (
        "Training and evaluation labels are "
        "provisional weak labels. Independent "
        "human and real-driving validation are required."
    ),
}

MOUTH_MODEL_CARD_PATH.write_text(
    json.dumps(
        mouth_model_card,
        indent=2,
    ) + "\n"
)

print(
    "Model card saved:",
    MOUTH_MODEL_CARD_PATH,
)

# --------------------------------------------------
# Bundle mouth artifacts
# --------------------------------------------------

mouth_artifact_paths = [
    MOUTH_CHECKPOINT_PATH,
    MOUTH_ONNX_PATH,
    MOUTH_MODEL_CARD_PATH,
    MOUTH_THRESHOLD_METADATA_PATH,
    MOUTH_TEST_METRICS_PATH,
    MOUTH_HISTORY_PATH,
    MOUTH_VALIDATION_PREDICTIONS_PATH,
    MOUTH_THRESHOLD_RESULTS_PATH,
    MOUTH_TEST_PREDICTIONS_PATH,
]

for path in mouth_artifact_paths:
    assert path.is_file(), (
        f"Missing artifact: {path}"
    )

with zipfile.ZipFile(
    MOUTH_BUNDLE_PATH,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as archive:
    for path in mouth_artifact_paths:
        archive.write(
            path,
            arcname=path.name,
        )

with zipfile.ZipFile(
    MOUTH_BUNDLE_PATH,
    "r",
) as archive:
    broken_entry = archive.testzip()

    assert broken_entry is None, (
        f"ZIP validation failed: {broken_entry}"
    )

# --------------------------------------------------
# SHA-256
# --------------------------------------------------

mouth_digest = hashlib.sha256()

with MOUTH_BUNDLE_PATH.open("rb") as handle:
    for chunk in iter(
        lambda: handle.read(
            4 * 1024 * 1024
        ),
        b"",
    ):
        mouth_digest.update(chunk)

mouth_bundle_sha256 = (
    mouth_digest.hexdigest()
)

MOUTH_CHECKSUM_PATH.write_text(
    f"{mouth_bundle_sha256}  "
    f"{MOUTH_BUNDLE_PATH.name}\n"
)

print()
print("Mouth model export completed")

print(
    "Checkpoint:",
    MOUTH_CHECKPOINT_PATH.name,
    f"{MOUTH_CHECKPOINT_PATH.stat().st_size / 1024**2:.2f} MiB",
)

print(
    "ONNX:",
    MOUTH_ONNX_PATH.name,
    f"{MOUTH_ONNX_PATH.stat().st_size / 1024**2:.2f} MiB",
)

print(
    "Bundle:",
    MOUTH_BUNDLE_PATH.name,
    f"{MOUTH_BUNDLE_PATH.stat().st_size / 1024**2:.2f} MiB",
)

print(
    "Bundle SHA-256:",
    mouth_bundle_sha256,
)

print("ZIP validation passed")

/tmp/ipykernel_58/499561004.py:73: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


ONNX export complete: /kaggle/working/drivealert_mouth_resnet18_v1.onnx
ONNX structural validation passed
Maximum PyTorch/ONNX logit difference: 5.4836273193359375e-06
PyTorch/ONNX parity validation passed
Model card saved: /kaggle/working/drivealert_mouth_resnet18_v1_model_card.json

Mouth model export completed
Checkpoint: drivealert_mouth_resnet18_v1_best.pth 42.72 MiB
ONNX: drivealert_mouth_resnet18_v1.onnx 42.63 MiB
Bundle: drivealert_mouth_resnet18_v1_artifacts.zip 79.34 MiB
Bundle SHA-256: f967e4c4d78058a75199d025b69ab25bd04ce08affa2a0133263d6047d63158b
ZIP validation passed
